# Phase 2 — Day 7: LogReg and Trees

**Date:** 2026-04-23  
**Phase:** 2 — Classical ML  
**Topic:** Logistic Regression and Decision Trees

---

## What you'll learn today
- How Logistic Regression turns a linear score into a probability
- What coefficients mean and how regularization (C) controls overfitting
- How Decision Trees split data using Gini impurity and entropy
- Why `max_depth` matters and how trees overfit without it


In [ ]:
# Setup — run this first
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.datasets import make_classification, load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler

import warnings
warnings.filterwarnings('ignore')
print('All imports OK!')


In [ ]:
# Sample data — binary classification: will a customer churn?
np.random.seed(42)
n = 300

age          = np.random.randint(18, 65, n)
tenure_months = np.random.randint(1, 60, n)
monthly_spend = np.random.uniform(20, 200, n)

# Churn is more likely for young customers with short tenure and high spend
log_odds = -2 + 0.03 * (30 - age) + 0.05 * (12 - tenure_months) + 0.01 * monthly_spend
prob_churn = 1 / (1 + np.exp(-log_odds))
churn = (np.random.uniform(0, 1, n) < prob_churn).astype(int)

df = pd.DataFrame({
    'age': age,
    'tenure_months': tenure_months,
    'monthly_spend': monthly_spend,
    'churn': churn
})

X = df[['age', 'tenure_months', 'monthly_spend']]
y = df['churn']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(df.head())
print(f'\nChurn rate: {y.mean():.1%}')
print(f'Train size: {len(X_train)}, Test size: {len(X_test)}')


---
## Part 1: Logistic Regression

Logistic Regression works in two steps. First it computes a linear score: `score = w1*x1 + w2*x2 + ... + b`. Then it squashes that score into a probability using the **sigmoid function**: `P = 1 / (1 + exp(-score))`. If P > 0.5, predict class 1.

The **coefficients** (`coef_`) tell you how much each feature pushes the log-odds up or down. A positive coefficient means the feature increases churn probability. A negative one decreases it.

**Regularization** prevents the model from fitting noise. The `C` parameter is the *inverse* of regularization strength — smaller C means more regularization (smaller weights, smoother model). Default is `C=1.0`.


In [ ]:
# Scale features — LogReg is sensitive to scale
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

# Fit Logistic Regression
lr = LogisticRegression(C=1.0, random_state=42)
lr.fit(X_train_sc, y_train)

# Look at coefficients
coef_df = pd.DataFrame({
    'feature': X.columns,
    'coefficient': lr.coef_[0]
}).sort_values('coefficient', ascending=False)

print('Coefficients (after scaling — comparable across features):')
print(coef_df.to_string(index=False))
print(f'\nIntercept: {lr.intercept_[0]:.4f}')
print(f'\nTest accuracy: {accuracy_score(y_test, lr.predict(X_test_sc)):.2%}')


In [ ]:
# How does C (regularization strength) affect accuracy?
C_values = [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]
results = []

for C in C_values:
    model = LogisticRegression(C=C, random_state=42, max_iter=1000)
    model.fit(X_train_sc, y_train)
    train_acc = accuracy_score(y_train, model.predict(X_train_sc))
    test_acc  = accuracy_score(y_test,  model.predict(X_test_sc))
    results.append({'C': C, 'train_acc': train_acc, 'test_acc': test_acc})

res_df = pd.DataFrame(results)
print(res_df.to_string(index=False))

# Key insight: very small C = underfitting, very large C = overfitting
print('\nSmall C = strong regularization = simpler model')
print('Large C = weak regularization = can overfit')


In [ ]:
# Getting probabilities, not just predictions
lr_best = LogisticRegression(C=1.0, random_state=42)
lr_best.fit(X_train_sc, y_train)

# predict_proba returns [prob_class_0, prob_class_1] for each sample
proba = lr_best.predict_proba(X_test_sc)

print('First 5 predictions:')
for i in range(5):
    print(f'  P(no churn)={proba[i,0]:.2%}  P(churn)={proba[i,1]:.2%}  -> predicted: {lr_best.predict(X_test_sc[i:i+1])[0]}')

# You can change the threshold for business reasons
# e.g., flag anyone with churn probability > 30%
threshold = 0.30
high_risk = (proba[:, 1] > threshold).astype(int)
print(f'\nUsing threshold {threshold}: {high_risk.sum()} customers flagged as high risk out of {len(high_risk)}')


---
## Part 2: Decision Trees

A Decision Tree splits data by asking yes/no questions about features. At each node, it picks the feature and threshold that best separates the classes. It measures "best" using **Gini impurity** or **entropy**.

**Gini impurity** = `1 - sum(p_i^2)`. It's 0 when a node is perfectly pure (all one class) and 0.5 when it's 50/50. It's fast to compute. **Entropy** = `-sum(p_i * log2(p_i))`. It comes from information theory and is slightly smoother but usually gives similar results.

The big danger with trees is **overfitting**. An unrestricted tree will grow until every leaf has one sample — perfect training accuracy, terrible test accuracy. `max_depth` is your main lever to stop this.


In [ ]:
# Decision Tree: Gini vs Entropy
for criterion in ['gini', 'entropy']:
    dt = DecisionTreeClassifier(criterion=criterion, max_depth=3, random_state=42)
    dt.fit(X_train, y_train)
    acc = accuracy_score(y_test, dt.predict(X_test))
    print(f'{criterion:>8}: test accuracy = {acc:.2%}')

# Print the tree structure (max_depth=3 so it's readable)
dt_gini = DecisionTreeClassifier(criterion='gini', max_depth=3, random_state=42)
dt_gini.fit(X_train, y_train)
print()
print(export_text(dt_gini, feature_names=list(X.columns)))


In [ ]:
# Overfitting demo: what happens as max_depth increases?
depths = range(1, 15)
train_accs = []
test_accs  = []

for d in depths:
    dt = DecisionTreeClassifier(max_depth=d, random_state=42)
    dt.fit(X_train, y_train)
    train_accs.append(accuracy_score(y_train, dt.predict(X_train)))
    test_accs.append(accuracy_score(y_test,  dt.predict(X_test)))

print('depth | train_acc | test_acc')
print('------+----------+---------')
for d, tr, te in zip(depths, train_accs, test_accs):
    marker = ' <-- sweet spot' if te == max(test_accs) else ''
    print(f'  {d:2d}  |  {tr:.2%}   |  {te:.2%}{marker}')

print(f'\nBest test accuracy at depth = {depths[test_accs.index(max(test_accs))]}')
print('Notice how train_acc eventually hits 100% while test_acc stops improving — that is overfitting.')


In [ ]:
# Feature importances from a Decision Tree
dt_best_depth = depths[test_accs.index(max(test_accs))]
dt_final = DecisionTreeClassifier(max_depth=dt_best_depth, random_state=42)
dt_final.fit(X_train, y_train)

importance_df = pd.DataFrame({
    'feature': X.columns,
    'importance': dt_final.feature_importances_
}).sort_values('importance', ascending=False)

print('Feature importances (how often + how much each feature reduces impurity):')
print(importance_df.to_string(index=False))
print('\nNote: importances sum to 1.0 (they are proportions, not percentages)')


---
## Tricky Bits — Common Mistakes

These are the mistakes that trip people up most often. Run the cells to see what breaks and why.


In [ ]:
# Mistake 1: Forgetting to scale before LogReg
# Without scaling, coefficients are not comparable across features

lr_unscaled = LogisticRegression(C=1.0, random_state=42, max_iter=1000)
lr_unscaled.fit(X_train, y_train)  # raw features

lr_scaled = LogisticRegression(C=1.0, random_state=42, max_iter=1000)
lr_scaled.fit(X_train_sc, y_train)  # scaled features

print('Unscaled coefficients:')
for feat, coef in zip(X.columns, lr_unscaled.coef_[0]):
    print(f'  {feat}: {coef:.6f}')

print('\nScaled coefficients:')
for feat, coef in zip(X.columns, lr_scaled.coef_[0]):
    print(f'  {feat}: {coef:.4f}')

print('\nUnscaled coefs are tiny for some features just because the feature has large values.')
print('Always scale before interpreting coefficients!')


In [ ]:
# Mistake 2: Confusing C with regularization strength
# C is the INVERSE of regularization: bigger C = LESS regularization

print('C parameter cheat sheet:')
print('  C=0.001  -> very strong regularization (simple model, may underfit)')
print('  C=1.0    -> default, balanced')
print('  C=1000   -> almost no regularization (complex model, may overfit)')

# Mistake 3: Unlimited tree depth on small datasets
dt_unlimited = DecisionTreeClassifier(random_state=42)  # no max_depth!
dt_unlimited.fit(X_train, y_train)

print(f'\nUnlimited tree: train={accuracy_score(y_train, dt_unlimited.predict(X_train)):.2%}, '
      f'test={accuracy_score(y_test, dt_unlimited.predict(X_test)):.2%}')
print(f'Tree depth reached: {dt_unlimited.get_depth()}')
print('Training accuracy is perfect but test accuracy is much worse. Classic overfitting.')


In [ ]:
# Mistake 4: Trees do NOT need feature scaling — LogReg does
# Trees only compare relative values (is x > threshold?), so scale does not matter

dt_scaled   = DecisionTreeClassifier(max_depth=3, random_state=42)
dt_unscaled = DecisionTreeClassifier(max_depth=3, random_state=42)

dt_scaled.fit(X_train_sc, y_train)
dt_unscaled.fit(X_train, y_train)

acc_s = accuracy_score(y_test, dt_scaled.predict(X_test_sc))
acc_u = accuracy_score(y_test, dt_unscaled.predict(X_test))

print(f'Tree on scaled data:   {acc_s:.4f}')
print(f'Tree on unscaled data: {acc_u:.4f}')
print('Same result. Trees are scale-invariant.')


---
## Trick Questions

Think before you open the answers!

---

**Q1.** You increase `C` from 1 to 100 in LogisticRegression. What happens to training accuracy? What about test accuracy?

<details><summary>Answer</summary>
Training accuracy usually goes up or stays the same (more freedom to fit the data). Test accuracy might go down if the model starts overfitting. Higher C = less regularization = more complex model.
</details>

---

**Q2.** A Decision Tree with no `max_depth` gets 100% training accuracy. Is this a good model?

<details><summary>Answer</summary>
Almost certainly not. The tree memorized every training sample (it grew until each leaf had one sample). This is severe overfitting. It will likely perform poorly on new data.
</details>

---

**Q3.** What does a Gini impurity of 0 mean for a node?

<details><summary>Answer</summary>
The node is perfectly pure — all samples at that node belong to the same class. No more splitting is needed.
</details>

---

**Q4.** You have features with very different scales (age: 18-65, salary: 20000-200000). Do you NEED to scale for a Decision Tree? For Logistic Regression?

<details><summary>Answer</summary>
Decision Tree: No. Trees split on thresholds and are scale-invariant. Logistic Regression: Yes (if you want to interpret coefficients or if regularization is involved, which it always is by default).
</details>

---

**Q5.** `feature_importances_` in a Decision Tree sum to what value? What do they represent?

<details><summary>Answer</summary>
They sum to 1.0. Each value represents the fraction of total impurity reduction that feature was responsible for across all splits in the tree.
</details>


---
## Exercises

Fill in the `___` parts. Run each cell — the `assert` will tell you if you got it right.


In [ ]:
# Exercise 1
# Fit a LogisticRegression with C=0.1 on the scaled training data.
# Store it in a variable called lr_ex1.

lr_ex1 = ___
___

assert hasattr(lr_ex1, 'coef_'), 'Model not fitted yet'
assert abs(lr_ex1.C - 0.1) < 1e-6, 'C should be 0.1'
print('Exercise 1 passed!')


In [ ]:
# Exercise 2
# Get the predicted probabilities from lr_ex1 on the scaled test set.
# Store the probability of churn (class 1) as a 1D array called churn_proba.

churn_proba = ___

assert churn_proba.shape == (len(X_test),), 'Should be a 1D array with one prob per test sample'
assert churn_proba.min() >= 0 and churn_proba.max() <= 1, 'Probabilities should be in [0, 1]'
print(f'Exercise 2 passed! Mean churn probability: {churn_proba.mean():.2%}')


In [ ]:
# Exercise 3
# Fit a DecisionTreeClassifier with criterion='entropy' and max_depth=4.
# Store it in dt_ex3. Train on the unscaled X_train.

dt_ex3 = ___
___

assert hasattr(dt_ex3, 'feature_importances_'), 'Model not fitted'
assert dt_ex3.criterion == 'entropy', 'criterion should be entropy'
assert dt_ex3.max_depth == 4, 'max_depth should be 4'
print(f'Exercise 3 passed! Test accuracy: {accuracy_score(y_test, dt_ex3.predict(X_test)):.2%}')


In [ ]:
# Exercise 4
# Get the feature importances from dt_ex3.
# Find the NAME of the most important feature. Store it as most_important.

most_important = ___

assert most_important in X.columns.tolist(), f'{most_important} is not a valid feature name'
print(f'Exercise 4 passed! Most important feature: {most_important}')


In [ ]:
# Exercise 5
# Manually calculate Gini impurity for a node with 40 positives and 60 negatives (100 total).
# Formula: gini = 1 - (p_pos^2 + p_neg^2)
# Store the result as gini_value.

p_pos = ___
p_neg = ___
gini_value = ___

assert abs(gini_value - 0.48) < 0.001, f'Expected ~0.48, got {gini_value:.4f}'
print(f'Exercise 5 passed! Gini = {gini_value:.4f}')


In [ ]:
# Exercise 6
# Find the best max_depth (from 1 to 10) for a DecisionTreeClassifier.
# Loop, fit, evaluate on X_test/y_test, and store the best depth as best_depth.

best_depth = None
best_acc   = 0

for d in range(1, 11):
    dt = DecisionTreeClassifier(max_depth=___, random_state=42)
    dt.fit(___, ___)
    acc = accuracy_score(___, dt.predict(___))
    if acc > best_acc:
        best_acc   = acc
        best_depth = ___

assert best_depth is not None, 'best_depth was never set'
assert 1 <= best_depth <= 10, 'best_depth should be between 1 and 10'
print(f'Exercise 6 passed! Best depth = {best_depth}, accuracy = {best_acc:.2%}')


In [ ]:
# Exercise 7
# Compare LogisticRegression (C=1, scaled) vs DecisionTree (max_depth=best_depth).
# Which one has higher test accuracy? Store 'logreg' or 'tree' as winner.

lr_final = LogisticRegression(C=1.0, random_state=42)
lr_final.fit(___, ___)
lr_acc = accuracy_score(___, lr_final.predict(___))

dt_final2 = DecisionTreeClassifier(max_depth=best_depth, random_state=42)
dt_final2.fit(___, ___)
dt_acc = accuracy_score(___, dt_final2.predict(___))

winner = ___ if lr_acc > dt_acc else ___

assert winner in ('logreg', 'tree'), 'winner must be logreg or tree'
print(f'Exercise 7 passed! LR: {lr_acc:.2%}, Tree: {dt_acc:.2%}, Winner: {winner}')


---
## Solutions

<details><summary>Show all solutions</summary>

```python
# Exercise 1
lr_ex1 = LogisticRegression(C=0.1, random_state=42)
lr_ex1.fit(X_train_sc, y_train)

# Exercise 2
churn_proba = lr_ex1.predict_proba(X_test_sc)[:, 1]

# Exercise 3
dt_ex3 = DecisionTreeClassifier(criterion='entropy', max_depth=4, random_state=42)
dt_ex3.fit(X_train, y_train)

# Exercise 4
most_important = X.columns[dt_ex3.feature_importances_.argmax()]

# Exercise 5
p_pos = 40 / 100
p_neg = 60 / 100
gini_value = 1 - (p_pos**2 + p_neg**2)

# Exercise 6
for d in range(1, 11):
    dt = DecisionTreeClassifier(max_depth=d, random_state=42)
    dt.fit(X_train, y_train)
    acc = accuracy_score(y_test, dt.predict(X_test))
    if acc > best_acc:
        best_acc   = acc
        best_depth = d

# Exercise 7
lr_final.fit(X_train_sc, y_train)
lr_acc = accuracy_score(y_test, lr_final.predict(X_test_sc))

dt_final2.fit(X_train, y_train)
dt_acc = accuracy_score(y_test, dt_final2.predict(X_test))

winner = 'logreg' if lr_acc > dt_acc else 'tree'
```
</details>


---
## Cumulative Review — Days 1-6

Mixed exercises covering pandas, numpy, data cleaning, Faker, PyTorch, and sklearn pipelines. Fill in the `___` blanks.


In [ ]:
# Cumulative 1 (Day 1 - Pandas)
# From the df DataFrame, get the mean monthly_spend for churned customers only.
# Store as mean_spend_churned.

mean_spend_churned = ___

assert isinstance(mean_spend_churned, float), 'Should be a float'
assert 50 < mean_spend_churned < 200, 'Seems out of range, double check'
print(f'Cumulative 1 passed! Mean spend for churned: {mean_spend_churned:.2f}')


In [ ]:
# Cumulative 2 (Day 2 - Numpy)
# Create a 1D numpy array of 50 evenly spaced values between 0 and 1.
# Then compute the mean and std of that array.

arr = ___
arr_mean = ___
arr_std  = ___

assert arr.shape == (50,), 'Should have 50 elements'
assert abs(arr_mean - 0.5) < 0.01, f'Mean should be ~0.5, got {arr_mean:.4f}'
print(f'Cumulative 2 passed! mean={arr_mean:.4f}, std={arr_std:.4f}')


In [ ]:
# Cumulative 3 (Day 3 - Data Cleaning)
# This DataFrame has missing values. Fill them with the column median.

dirty = pd.DataFrame({
    'score': [10, None, 30, None, 50],
    'days':  [1,  2,    None, 4,  5]
})

cleaned = ___

assert cleaned.isnull().sum().sum() == 0, 'Still has missing values'
assert cleaned.loc[1, 'score'] == 30.0, 'score median should be 30'
print('Cumulative 3 passed! No more missing values.')


In [ ]:
# Cumulative 4 (Day 4 - Python Core)
# Use a dict comprehension to create a dict mapping each feature name to its max value in X.
# Store it as feature_maxes.

feature_maxes = {___: ___ for ___ in X.columns}

assert set(feature_maxes.keys()) == set(X.columns), 'Keys should be feature names'
assert feature_maxes['age'] == X['age'].max(), 'Wrong max for age'
print(f'Cumulative 4 passed! Feature maxes: {feature_maxes}')


In [ ]:
# Cumulative 5 (Day 5 - PyTorch)
import torch

# Create a 1D tensor of the first 5 churn labels (from y_train).
# Then compute the mean as a Python float.

labels_tensor = ___
labels_mean   = ___

assert isinstance(labels_tensor, torch.Tensor), 'Should be a tensor'
assert labels_tensor.shape == (5,), 'Should have 5 elements'
assert isinstance(labels_mean, float), 'Mean should be a Python float'
print(f'Cumulative 5 passed! Tensor: {labels_tensor}, mean: {labels_mean}')


In [ ]:
# Cumulative 6 (Day 6 - Pipelines)
from sklearn.pipeline import Pipeline

# Build a Pipeline: StandardScaler -> LogisticRegression(C=1).
# Fit it on X_train/y_train, score on X_test/y_test.

pipe = Pipeline(steps=[
    ('scaler', ___),
    ('model',  ___)
])

pipe.fit(___, ___)
pipe_score = pipe.score(___, ___)

assert 0.5 < pipe_score < 1.0, f'Score looks wrong: {pipe_score}'
print(f'Cumulative 6 passed! Pipeline accuracy: {pipe_score:.2%}')


---
## Cumulative Solutions

<details><summary>Show solutions</summary>

```python
# Cumulative 1
mean_spend_churned = df[df['churn'] == 1]['monthly_spend'].mean()

# Cumulative 2
arr = np.linspace(0, 1, 50)
arr_mean = arr.mean()
arr_std  = arr.std()

# Cumulative 3
cleaned = dirty.fillna(dirty.median())

# Cumulative 4
feature_maxes = {col: X[col].max() for col in X.columns}

# Cumulative 5
labels_tensor = torch.tensor(y_train.values[:5].astype(float))
labels_mean   = labels_tensor.mean().item()

# Cumulative 6
pipe = Pipeline(steps=[
    ('scaler', StandardScaler()),
    ('model',  LogisticRegression(C=1.0, random_state=42))
])
pipe.fit(X_train, y_train)
pipe_score = pipe.score(X_test, y_test)
```
</details>


In [ ]:
# Cheat Sheet — Day 7
cheat = """
=== LOGISTIC REGRESSION ===
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(C=1.0, penalty='l2', max_iter=1000)
lr.fit(X_train_scaled, y_train)

lr.coef_          -> coefficients (one per feature, scaled = comparable)
lr.intercept_     -> bias term
lr.predict(X)     -> class labels
lr.predict_proba(X)[:, 1]  -> probability of class 1

C param: INVERSE regularization strength
  C=0.01  -> strong regularization (small weights)
  C=1.0   -> default
  C=100   -> weak regularization (can overfit)

ALWAYS scale features before LogReg!

=== DECISION TREE ===
from sklearn.tree import DecisionTreeClassifier, export_text

dt = DecisionTreeClassifier(
    criterion='gini',    # or 'entropy'
    max_depth=5,         # control overfitting
    min_samples_split=2, # min samples to split a node
    random_state=42
)
dt.fit(X_train, y_train)  # no scaling needed

dt.feature_importances_    -> array summing to 1.0
dt.get_depth()             -> actual depth reached
export_text(dt, feature_names=list(X.columns))  -> print tree

Gini impurity: 1 - sum(p_i^2)  -> 0 = pure, 0.5 = 50/50
Entropy:  -sum(p_i * log2(p_i)) -> 0 = pure

=== KEY DIFFERENCES ===
              LogReg      Decision Tree
Needs scaling   YES           NO
Interpretable  Coefs       Tree structure
Overfits via   Large C     Deep tree
Probabilities  Yes (smooth) Yes (less reliable)
"""
print(cheat)


---
## You did Day 7!

**Next up: Day 8 — EnsemblesAndXGBoost**

You will learn how Random Forest and XGBoost improve on a single Decision Tree by combining many trees together. Bagging vs boosting, feature importance at scale, and your first gradient boosting model.
